In [1]:
from pathlib import Path
import os 
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
import unicodedata
from rapidfuzz import process
from clase_reportes_new import ReportClassNew
from send_mail import MailSender
rc = ReportClassNew()
ms = MailSender()


In [5]:
carpeta = rc.consolidar_carpeta(ruta_carpeta=r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA', extension='csv', sep=';')

Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA
  - Archivo 'Ventas_Junio_2024_Diciembre_2024.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2025_Diciembre_2025.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2026_Marzo_2026.csv' leído correctamente.
  - Archivo 'Ventas_Abril_2026.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [6]:
carpeta.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'DIA', 'MES', 'AÑO', 'CLIENTE',
       'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
       'TRM', 'TOTAL($)', 'TOTAL_CON_IMPUESTOS', 'TELEFONO', 'EMAIL', 'PAIS',
       'CIUDAD_x', 'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS',
       'REFERENCIA', 'ASESOR COMERCIAL', 'CIUDAD_y', 'ZONA', 'CLIENTE_ORI'],
      dtype='object')

In [8]:
carpeta[['CLIENTE', 'CATEGORÍA']].drop_duplicates().to_excel(r'D:\Desktop\categoria_clientes.xlsx', index=False )

In [ ]:
prueba = rc.pipeline_bi()

OK

In [ ]:
ventas_procesadas = rc.transformar_base()
ruta = rc.validar_ruta()

ruta_clean = ruta / 'CLEAN DATA' 


ruta2 = Path(ventas_procesadas['nombre_archivo'])
ruta_carpeta = ruta_clean / ruta2
ruta_errores = ruta / 'file' / 'ventas_sin_categoria.csv'
ruta_padres = ruta / 'data' / 'clientes_padres.xlsx'
ruta_bgta= ruta / 'data' / 'Base_bogota.xlsx'
ruta_zonas= ruta / 'data' / 'zonas.xlsx'
ruta_zonas_cundi = ruta / 'data' / 'zonas_cundinamarca.xlsx'
ruta_kits = ruta / 'data' / 'kits.xlsx'
df_kits = pd.read_excel(ruta_kits)
df_padres = pd.read_excel(ruta_padres)
df_bogota= pd.read_excel(ruta_bgta)
df_bogota = df_bogota.drop_duplicates(subset='DOCUMENTO')
df_bogota['DOCUMENTO'] = df_bogota['DOCUMENTO'].astype(int)
zonas = pd.read_excel(ruta_zonas)
cundinamarca = pd.read_excel(ruta_zonas_cundi)
ventas_procesadas['Base'] =  ventas_procesadas['Base'].merge(zonas, on=['DEPARTAMENTO', 'CATEGORÍA'], how='left')\
                        .merge(cundinamarca,  on=['DEPARTAMENTO','CIUDAD', 'CATEGORÍA'], how='left')


ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'] = pd.to_numeric(
    ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'], errors='coerce'
).astype('float')

df_bogota['DOCUMENTO'] = pd.to_numeric(
    df_bogota['DOCUMENTO'], errors='coerce'
).astype('float')


ventas_procesadas['Base'].loc[
    (ventas_procesadas['Base']['DEPARTAMENTO'] == 'Cundinamarca (CO)') &
    (ventas_procesadas['Base']['CATEGORÍA'] == 'MAYORISTA NV') &
    (ventas_procesadas['Base']['zona'] == 'sin zona') &
    (ventas_procesadas['Base']['ZONA_CUNDINAMARCA'].notna()),
    'zona'
] = ventas_procesadas['Base']['ZONA_CUNDINAMARCA']

ventas_procesadas['Base'] = ventas_procesadas['Base'].merge(df_bogota[['DOCUMENTO', 'CATEGORÍA', 'ZONA']], 
                                left_on=['IDENTIFICACION_CLIENTE','CATEGORÍA'], right_on=['DOCUMENTO', 'CATEGORÍA'], how='left')

ventas_procesadas['Base']['ZONA'] = ventas_procesadas['Base']['ZONA'].fillna(ventas_procesadas['Base']['zona'])

ventas_procesadas['Base'] = ventas_procesadas['Base'].drop(columns=['zona', 'DOCUMENTO', 'ZONA_CUNDINAMARCA'])
df_padres.drop_duplicates(subset='CLIENTE', inplace=True)
ventas_procesadas['Base']['CLIENTE_ORI'] = ventas_procesadas['Base']['CLIENTE'] 
df_padres = df_padres[['CLIENTE', 'CLIENTE PADRE']]
ventas_procesadas['Base'] = ventas_procesadas['Base'].merge(df_padres, on='CLIENTE', how='left')
ventas_procesadas['Base']['CLIENTE PADRE'] = ventas_procesadas['Base']['CLIENTE PADRE'].fillna(ventas_procesadas['Base']['CLIENTE'])
ventas_procesadas['Base']['CLIENTE'] = ventas_procesadas['Base']['CLIENTE PADRE'] 
ventas_procesadas['Base'].drop(columns=['CLIENTE PADRE'], inplace=True)

try:
    ruta_clean.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_clean}' creada o ya existe.")
    ventas_procesadas['Base'].to_csv(ruta_carpeta, index=False, sep=';', encoding='utf-8-sig', decimal=',')
    ruta_exploded = ruta / 'exploded_data'
    base_exploded = rc.explosion_ventas(df_ventas=ventas_procesadas['Base'], df_kits=df_kits)
    ruta_exploded.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_clean}' creada o ya existe.")
    base_exploded['base'].to_csv(ruta_exploded / f"exploded_{ruta2.name}", index=False, sep=';', encoding='utf-8-sig', decimal=',')
    with pd.ExcelWriter(ruta_errores, engine='openpyxl') as writer:
        ventas_procesadas['errores'].to_excel(writer, sheet_name='etiqueta a tipo', index=False)
        ventas_procesadas['cliente_call_center'].to_excel(writer, sheet_name='CLIENTE a CALL', index=False)
        # ventas_procesadas['asesores_sin_categoria'].to_excel(writer, sheet_name='Mayoristas sin categoria', index=False)
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")
    


# Consolidar ventas
base_clean = rc.consolidar_carpeta(ruta_carpeta=ruta_clean, extension='csv')
ruta_base = ruta / 'file' / 'base_ventas.csv'
import locale
try:         
    # Intentamos usar el locale en español para obtener "ENERO", "FEBRERO", etc.
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except locale.Error:
    print("   - Advertencia: Locale 'es_ES.UTF-8' no disponible. Se usarán nombres de mes en inglés.")
    
base_clean['FECHA_FACTURA'] = pd.to_datetime(base_clean['FECHA_FACTURA'])
base_clean['MES'] = base_clean['FECHA_FACTURA'].dt.strftime('%B').str.upper()
columnas_finales = [
        "Source.Name", "NUMERO_FACTURA", "FECHA_FACTURA", "AÑO", "MES", "DIA",
        "CLIENTE", "IDENTIFICACION_CLIENTE", "CATEGORÍA", "PRODUCTO", "CANTIDAD",
        "TOTAL", "TASA_CAMBIO", "TRM", "TOTAL($)", "TOTAL_CON_IMPUESTOS" "TELEFONO", "EMAIL", "PAIS",
        "CIUDAD", "CIUDAD_CORREGIDA", "DEPARTAMENTO", "EQUIPO_VENTAS", "REFERENCIA", "ZONA" , "CLIENTE_ORI"
    ]
    
# Manejo defensivo por si la columna 'ASESOR COMERCIAL' no siempre existe
if 'ASESOR COMERCIAL' in base_clean.columns:
    base_clean['ASESOR COMERCIAL'] = base_clean['ASESOR COMERCIAL'].astype(str)
    columnas_finales.append('ASESOR COMERCIAL')

# Aseguramos que solo reordenamos las columnas que realmente existen en el DataFrame
columnas_existentes = [col for col in columnas_finales if col in base_clean.columns]
base_clean = base_clean[columnas_existentes]

# Esta linea mantiene solo los pruductos comerciales
base_clean = base_clean[base_clean['PRODUCTO'].str.startswith(('[PCN','[KD','[TNG','[B8'))]   ###### linea modificada 


try:
    ruta_file = ruta / 'file' 
    ruta_file.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_file}' creada o ya existe.")
    base_clean.to_csv(ruta_base, index=False, sep=';', encoding='utf-8-sig', decimal=',')
    
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")

    
# Consolidar ventas
base_exploded = rc.consolidar_carpeta(ruta_carpeta=ruta_exploded, extension='csv')
ruta_base = ruta / 'file' / 'base_ventas_exploded.csv'
import locale
try:         
    # Intentamos usar el locale en español para obtener "ENERO", "FEBRERO", etc.
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except locale.Error:
    print("   - Advertencia: Locale 'es_ES.UTF-8' no disponible. Se usarán nombres de mes en inglés.")


try:
    ruta_file = ruta / 'file' 
    ruta_file.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_file}' creada o ya existe.")
    base_exploded.to_csv(ruta_base, index=False, sep=';', encoding='utf-8-sig', decimal=',')
    
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")

    
explosion = rc.explosion_ventas(df_ventas=base_clean, df_kits=None)
return {'ventas_procesadas':ventas_procesadas,
        'base_clean':base_clean,
        'explosion':base_exploded
        }
